# BirdCLEF+ 2026 - Submission Notebook

**Environment:** Kaggle CPU notebook (no GPU, no internet, 90-min limit)

**Required Kaggle Datasets to attach:**
1. `birdclef-2026` - competition data
2. `birdclef-baseline-model` - trained model .pth file
3. `birdclef-src` - source code (src/ and config/ directories)

In [ ]:
import os
import sys
import time

import numpy as np
import pandas as pd
import torch
import librosa
from tqdm.auto import tqdm

START_TIME = time.time()

# Detect environment
ON_KAGGLE = os.path.exists("/kaggle/input")

if ON_KAGGLE:
    KAGGLE_USER = "stochastix"
    DATA_DIR = "/kaggle/input/competitions/birdclef-2026"
    MODEL_PATH = f"/kaggle/input/datasets/{KAGGLE_USER}/birdclef-baseline-model/best_model.pth"
    SRC_DIR = f"/kaggle/input/datasets/{KAGGLE_USER}/birdclef-src"
    sys.path.insert(0, SRC_DIR)
else:
    DATA_DIR = "../data"
    MODEL_PATH = "../models/best_model.pth"
    sys.path.insert(0, "..")

TEST_DIR = os.path.join(DATA_DIR, "test_soundscapes")
TAXONOMY_CSV = os.path.join(DATA_DIR, "taxonomy.csv")
SAMPLE_SUB = os.path.join(DATA_DIR, "sample_submission.csv")

print(f"ON_KAGGLE: {ON_KAGGLE}")
print(f"TEST_DIR: {TEST_DIR}")
print(f"MODEL_PATH: {MODEL_PATH}")
print(f"Model file exists: {os.path.exists(MODEL_PATH)}")

In [ ]:
import yaml
from src.audio import load_audio, audio_to_melspec, normalize_melspec
from src.transforms import spectrogram_to_tensor
from src.dataset import load_taxonomy
from src.model import get_model

# Load config
config_path = os.path.join(SRC_DIR, "config", "default.yaml") if ON_KAGGLE else "../config/default.yaml"
with open(config_path) as f:
    config = yaml.safe_load(f)

audio_config = config["audio"]
SR = audio_config["sample_rate"]
DURATION = audio_config["duration"]
IN_CHANNELS = config["model"].get("in_channels", 1)
BATCH_SIZE = config["inference"].get("batch_size", 16)

# Load species list
species_list = load_taxonomy(TAXONOMY_CSV)
print(f"Species: {len(species_list)}")

## Load Model

In [ ]:
device = torch.device("cpu")

model = get_model(config["model"])
model.load_state_dict(torch.load(MODEL_PATH, map_location=device, weights_only=True))
model.eval()

params = sum(p.numel() for p in model.parameters())
print(f"Model loaded: {params:,} parameters on {device}")
print(f"Time elapsed: {time.time() - START_TIME:.1f}s")

## Enumerate Test Windows

In [ ]:
import librosa

test_files = sorted([f for f in os.listdir(TEST_DIR) if f.endswith(".ogg")])
print(f"Test soundscapes: {len(test_files)}")

# Build list of all windows
windows = []
for filename in test_files:
    file_path = os.path.join(TEST_DIR, filename)
    total_dur = librosa.get_duration(path=file_path)
    basename = os.path.splitext(filename)[0]

    start = 0.0
    while start + DURATION <= total_dur + 0.01:
        end = start + DURATION
        row_id = f"{basename}_{int(end)}"
        windows.append({
            "file_path": file_path,
            "start": start,
            "row_id": row_id,
        })
        start += DURATION

print(f"Total windows: {len(windows)}")
print(f"Time elapsed: {time.time() - START_TIME:.1f}s")

## Run Inference

In [ ]:
all_predictions = []

# Process in batches
for batch_start in tqdm(range(0, len(windows), BATCH_SIZE), desc="Inference"):
    batch_windows = windows[batch_start:batch_start + BATCH_SIZE]
    batch_tensors = []
    batch_row_ids = []

    for w in batch_windows:
        audio = load_audio(w["file_path"], sr=SR, duration=DURATION, offset=w["start"])
        melspec = audio_to_melspec(audio, SR, audio_config)
        melspec = normalize_melspec(melspec)
        tensor = spectrogram_to_tensor(melspec, IN_CHANNELS)
        batch_tensors.append(tensor)
        batch_row_ids.append(w["row_id"])

    # Stack and predict
    batch_input = torch.stack(batch_tensors)
    with torch.no_grad():
        logits = model(batch_input)
        probs = torch.sigmoid(logits).numpy()

    for row_id, prob_vec in zip(batch_row_ids, probs):
        all_predictions.append((row_id, prob_vec))

print(f"Predictions: {len(all_predictions)}")
print(f"Time elapsed: {time.time() - START_TIME:.1f}s")

## Create Submission

In [ ]:
# Build submission DataFrame
rows = []
for row_id, probs in all_predictions:
    row = {"row_id": row_id}
    for species, prob in zip(species_list, probs):
        row[species] = float(prob)
    rows.append(row)

submission = pd.DataFrame(rows)

# Ensure column order matches sample submission
if os.path.exists(SAMPLE_SUB):
    sample = pd.read_csv(SAMPLE_SUB, nrows=1)
    submission = submission[sample.columns]

# Save
submission.to_csv("submission.csv", index=False)
print(f"Submission shape: {submission.shape}")
print(f"Columns: {len(submission.columns)} (1 row_id + {len(submission.columns) - 1} species)")
submission.head()

## Sanity Checks

In [ ]:
# Verify submission
assert submission.shape[1] == len(species_list) + 1, f"Expected {len(species_list) + 1} columns, got {submission.shape[1]}"
assert not submission.isnull().any().any(), "Submission contains NaN values!"

prob_cols = submission.columns[1:]
prob_values = submission[prob_cols].values
assert prob_values.min() >= 0, f"Min probability is {prob_values.min()}"
assert prob_values.max() <= 1, f"Max probability is {prob_values.max()}"

print("All checks passed!")
print(f"Rows: {len(submission)}")
print(f"Probability range: [{prob_values.min():.6f}, {prob_values.max():.6f}]")
print(f"Mean probability: {prob_values.mean():.6f}")
print(f"\nTotal runtime: {time.time() - START_TIME:.1f}s")